Text wrapper (for sane human experience reading this notebook)

In [2]:
from IPython.display import HTML, display

def wrap_outputs():
    display(HTML('''
    <style>
        pre {
            white-space: pre-wrap !important;
            word-wrap: break-word !important;
        }
    </style>
    '''))

# This registers the function to run silently in the background
# right before any cell executes, ensuring the CSS is always applied.
get_ipython().events.register('pre_run_cell', wrap_outputs)

# Install agent

In [3]:
!pip install google-genai pydantic

In [4]:
import os
from google.colab import userdata
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# 1. Authenticate securely using the Colab Secret
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
client = genai.Client()

In [8]:
from pydantic import BaseModel, Field
from google import genai
from google.genai import types
import time
import json

# 1. Define the JSON structure
class EvaluationReport(BaseModel):
    certainty_rate: int = Field(description="Integer from 1 to 100 representing how well the text meets the criteria.")
    review_report: str = Field(description="A concise literary critique justifying the score based on the quality of prose and characterisation and alignment with the topic.")
    extracted_text: str = Field(description="The exact text of the relevant ghost story extracted from the magazine. If rejected or no ghost story is found, leave empty.")

# 2. The System Prompt (The Gothic Editor Persona)
system_instruction = """
You are an expert literary editor specializing in gothic fiction and the supernatural.
Your task is to analyze the provided text snippet from a literary magazine, determine if it contains a ghost story, evaluate its quality, and EXTRACT that specific story.

Evaluate the text against the following parameters:
1. Genre (Ghost Story): the text must be a fictional story featuring ghosts, hauntings, or explicitly supernatural elements.
2. Extraction: you must copy the exact text of the relevant ghost story into the 'extracted_text' JSON field. Ignore all essays, poems, or unrelated stories in the magazine.
3. Prose quality: the writing must demonstrate high-quality literary craftsmanship, effective atmospheric tension and strong characterisation.

Return a strict evaluation. A certainty_rate of 90 or above means the text contains an exceptional ghost story that perfectly meets all criteria.
"""

# 3. The Execution Function (With Exponential Backoff for Rate Limits)
def evaluate_text(magazine_text, retries=3):
    print("Reviewer is scanning the magazine issue for supernatural fiction...")

    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash-lite',
                contents=magazine_text,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    response_mime_type="application/json",
                    response_schema=EvaluationReport,
                    temperature=0.2 # Low temperature for strict evaluation and precise extraction
                ),
            )
            return response.text

        except Exception as e:
            error_message = str(e)
            if '429' in error_message or 'RESOURCE_EXHAUSTED' in error_message:
                wait_time = 60 * (attempt + 1)
                print(f"API Rate limit hit. Cooling down for {wait_time} seconds before retrying...")
                time.sleep(wait_time)
            else:
                print(f"An unexpected API error occurred: {error_message}")
                return '{"certainty_rate": 0, "review_report": "System Error: Evaluation failed.", "extracted_text": ""}'

    print("Reviewer failed after multiple retries.")
    return '{"certainty_rate": 0, "review_report": "System Error: Max retries exceeded.", "extracted_text": ""}'

# --- TESTING THE REVIEWER ---
# Run a quick test to ensure the JSON validates properly
sample_magazine_text = """
The London Literary Gazette - October Issue.
[Poem omitted]
THE PHANTOM COACH by Amelia B. Edwards.
The circumstances I am about to relate to you have truth to recommend them. They happened to myself, and my recollection of them is as vivid as if they had taken place only yesterday...
[End of story]
An Essay on the Modern Trade Winds...
"""

result = evaluate_text(sample_magazine_text)
print(result)

Reviewer is scanning the magazine issue for supernatural fiction...
{
  "certainty_rate": 95,
  "review_report": "The story 'The Phantom Coach' by Amelia B. Edwards is a classic ghost story, featuring a personal account of a supernatural encounter. The prose is evocative, building suspense effectively, and the narrative voice is compelling, drawing the reader into the author's chilling experience. It aligns perfectly with the gothic and supernatural genre.",
  "extracted_text": "The circumstances I am about to relate to you have truth to recommend them. They happened to myself, and my recollection of them is as vivid as if they had taken place only yesterday..."
}


# Python script to search for lit

In [5]:
!pip install beautifulsoup4 requests pandas

In [6]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import json
import os

*Search for links*

In [9]:
# 1. Establish Session
session = requests.Session()
session.cookies.set('view_adult', 'true', domain='gutenberg.org')

approved_works_ledger = "approved_works_queue.csv"

# 2. The Text Extractor
def scrape_gutenberg_work(work_url):
    print(f"Approaching Gutenberg politely: {work_url}")
    time.sleep(3) # Standard polite pause

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

    # 1. Fetch the metadata (Title)
    try:
        response = requests.get(work_url, headers=headers, timeout=45)
        if response.status_code != 200:
            print("Failed to access book page.")
            return None

        soup = BeautifulSoup(response.content, 'html.parser')
        # Gutenberg titles are stored in an h1 with itemprop="name"
        title_tag = soup.find('h1', itemprop='name')
        title = title_tag.text.strip() if title_tag else "Unknown Title"

    except requests.exceptions.RequestException as e:
        print(f"Metadata request failed: {e}")
        return None

    # 2. Fetch the raw text directly
    text_url = f"{work_url}.txt.utf-8"
    try:
        text_response = requests.get(text_url, headers=headers, timeout=45)
        if text_response.status_code == 200:
            full_text = text_response.text
        else:
            print(f"Plain text file not found at {text_url}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Text extraction failed: {e}")
        return None

    return {"title": title, "url": work_url, "text": full_text}

# 3. The Evaluator & Ledger Manager
def process_pipeline(work_url):
    story_data = scrape_gutenberg_work(work_url)

    if not story_data or len(story_data['text']) < 100:
        print("Could not extract meaningful text.")
        return

    text_to_analyse = story_data['text'][:50000]

    # Note: This relies on the 'evaluate_text' function
    evaluation_json = evaluate_text(text_to_analyse)

    try:
        eval_data = json.loads(evaluation_json)
        certainty = eval_data.get('certainty_rate', 0)
        report = eval_data.get('review_report', '')
        extracted = eval_data.get('extracted_text', '')

        print(f"\nEvaluating: {story_data['title']}")
        print(f"Score: {certainty}%")
        print(f"Report: {report}")

        if os.path.exists(approved_works_ledger):
            df = pd.read_csv(approved_works_ledger)
        else:
            df = pd.DataFrame(columns=["Title", "URL", "Score", "Report", "Extracted_Text"])

        url_exists = story_data['url'] in df['URL'].values

        if certainty >= 90:
            print(">>> SUCCESS: Work meets strict criteria.")
            if url_exists:
                print(">>> UPDATING: Overwriting old score in the ledger.")
                df.loc[df['URL'] == story_data['url'], ['Score', 'Report', 'Extracted_Text', 'Title']] = [certainty, report, extracted, story_data['title']]
            else:
                print(">>> ADDING: Writing new work to the translation queue.")
                new_row = pd.DataFrame([{"Title": story_data['title'], "URL": story_data['url'], "Score": certainty, "Report": report, "Extracted_Text": extracted}])
                df = pd.concat([df, new_row], ignore_index=True)

            df.to_csv(approved_works_ledger, index=False)

        else:
            print(">>> REJECTED: Work did not meet the 90% threshold.")
            if url_exists:
                print(f">>> PURGING: Removing previously approved work from the queue due to new score.")
                df = df[df['URL'] != story_data['url']]
                df.to_csv(approved_works_ledger, index=False)

    except json.JSONDecodeError:
        print("Reviewer failed to return valid JSON.")

# 4. The URL Harvester
def harvest_search_results(search_url, max_works=5):
    print(f"Scraping Gutenberg search page: {search_url}")
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

    try:
        response = requests.get(search_url, headers=headers, timeout=45)
    except requests.exceptions.RequestException as e:
        print(f"Search page request failed: {e}")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    work_links = []

    # Gutenberg lists books under the 'booklink' class
    works = soup.find_all('li', class_='booklink')

    for work in works[:max_works]:
        a_tag = work.find('a', class_='link')
        if a_tag and '/ebooks/' in a_tag['href']:
            full_url = "https://www.gutenberg.org" + a_tag['href']
            work_links.append(full_url)

    print(f"Harvester found {len(work_links)} works to pass to Reviewer.")
    return work_links

# 5. The Autonomous Loop
def run_autonomous_harvester(search_url):
    urls_to_process = harvest_search_results(search_url, max_works=5)

    for url in urls_to_process:
        print(f"\n--- Initiating Pipeline for: {url} ---")
        process_pipeline(url)

        print("Cooling down systems for 20 seconds...")
        time.sleep(20)

# --- EXECUTION ---
filtered_search_url = "https://www.gutenberg.org/ebooks/bookshelf/298"

run_autonomous_harvester(filtered_search_url)

Scraping Gutenberg search page: https://www.gutenberg.org/ebooks/bookshelf/298
Harvester found 5 works to pass to Reviewer.

--- Initiating Pipeline for: https://www.gutenberg.org/ebooks/10020 ---
Approaching Gutenberg politely: https://www.gutenberg.org/ebooks/10020
Reviewer is scanning the magazine issue for supernatural fiction...

Evaluating: The Strand Magazine: Vol. 07, Issue 37, January, 1894. by Various
Score: 95%
Report: This story is an excellent example of gothic supernatural fiction. The prose is evocative, building a palpable sense of dread and mystery. The characterisation, particularly of the increasingly desperate Lady Studley and the tormented Sir Henry, is compelling. The narrative skillfully weaves elements of psychological horror with a genuine ghostly manifestation, culminating in a shocking and satisfying twist that aligns perfectly with the genre's conventions.
>>> SUCCESS: Work meets strict criteria.
>>> ADDING: Writing new work to the translation queue.
Cooling

Human selection step (sign-off one story for translation)

In [10]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

master_queue_file = "approved_works_queue.csv"
# We create a new specific file just for Translator to read from
translation_target_file = "selected_for_translation.csv"

if not os.path.exists(master_queue_file):
    print("The master queue is empty. Run the Harvester first.")
else:
    df = pd.read_csv(master_queue_file)

    # Filter for only the highly rated works
    df['Score'] = pd.to_numeric(df['Score'], errors='coerce')
    df = df[df['Score'] >= 90]

    if df.empty:
        print("No works in the master queue meet the 90% threshold.")
    else:
        print("Human Intercept Required: Select a single work to translate to conserve tokens.\n")

        # Create a dictionary mapping the visual title/score to the actual URL
        options = {f"{row['Title']} (Score: {row['Score']}%)": row['URL'] for index, row in df.iterrows()}

        # Create the interactive UI elements
        dropdown = widgets.Dropdown(
            options=options,
            description='Select Book:',
            layout=widgets.Layout(width='80%')
        )

        confirm_button = widgets.Button(
            description='Confirm Selection',
            button_style='info',
            icon='check'
        )

        output = widgets.Output()

        # The function that runs when you click Confirm
        def on_confirm_clicked(b):
            with output:
                clear_output() # Clears previous messages
                selected_url = dropdown.value

                # Isolate the exact row the user chose
                selected_row = df[df['URL'] == selected_url]

                # Write ONLY this single row to the new target file
                selected_row.to_csv(translation_target_file, index=False)

                title_locked = selected_row['Title'].values[0]
                print(f"🔒 SUCCESS: '{title_locked}' has been isolated for translation.")
                print(">>> You may now run the Translator.")

        # Bind the click event and display the UI
        confirm_button.on_click(on_confirm_clicked)
        display(dropdown, confirm_button, output)

Human Intercept Required: Select a single work to translate to conserve tokens.



Dropdown(description='Select Book:', layout=Layout(width='80%'), options={'The Strand Magazine: Vol. 07, Issue…

Button(button_style='info', description='Confirm Selection', icon='check', style=ButtonStyle())

Output()

# The translator

In [ ]:
!pip install EbookLib
from ebooklib import epub
import pandas as pd
import time
import os
from google.colab import files
import ipywidgets as widgets
from IPython.display import display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.6 MB/s eta 0:00:00


In [ ]:
# 1. The Glossary
book_glossary = """
- Ghost: Gespenst
- History: Geschichte
- Shadow: Schatten
"""

# 2. The Translator
def translate_text(text_chunk):
    print("Translating chunk...")
    system_instruction = f"""
    You are an expert literary translator specialising in the English to German translation direction.
    Translate the following text from English to German.

    Crucial Directives:
    1. Maintain the literary style and atmospheric tension of the original prose.
    2. Avoid abridging and summarisation. Translate every sentence faithfully.
    3. Adhere to this specific glossary:
    {book_glossary}
    """

    # We use Flash-Lite here as it is fast and budget-friendly
    response = client.models.generate_content(
        model='gemini-2.5-flash-lite',
        contents=text_chunk,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.3 # Slight creativity allowed for atmospheric literary flow
        ),
    )
    return response.text

# 3. The Chunker
def chunk_text(text, max_chars=4000):
    # formatting fallback: try double newlines, if that fails, use single newlines
    paragraphs = text.split('\n\n')
    if len(paragraphs) < 3:
        paragraphs = text.split('\n')

    chunks = []
    current_chunk = ""

    for p in paragraphs:
        if len(current_chunk) + len(p) < max_chars:
            current_chunk += p + "\n"
        else:
            chunks.append(current_chunk.strip())
            current_chunk = p + "\n"
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

# 4. The Publisher (Creating the EPUB with Download Button)
def publish_to_epub(title, translated_chunks):
    print(f"\nBinding '{title}' into EPUB...")
    book = epub.EpubBook()

    book.set_identifier(f"gutenberg_translated_{title.replace(' ', '_')}")
    book.set_title(title)
    book.set_language('de')

    chapters = []
    for i, chunk in enumerate(translated_chunks):
        c = epub.EpubHtml(title=f'Part {i+1}', file_name=f'part_{i+1}.xhtml', lang='de')
        html_content = "".join([f"<p>{p}</p>" for p in chunk.split('\n')])
        c.content = f"<h2>Part {i+1}</h2>{html_content}"
        book.add_item(c)
        chapters.append(c)

    book.toc = tuple(chapters)
    book.add_item(epub.EpubNcx())
    book.add_item(epub.EpubNav())

    spine = ['nav'] + chapters
    book.spine = spine

    safe_title = "".join([c for c in title if c.isalpha() or c.isdigit() or c==' ']).rstrip()
    filename = f"{safe_title.replace(' ', '_')}.epub"
    epub.write_epub(filename, book, {})
    print(f"SUCCESS: Saved as {filename}")

    # --- THE INTERACTIVE DOWNLOAD BUTTON ---
    download_button = widgets.Button(
        description=f"📥 Download {safe_title[:15]}...",
        button_style='success',
        tooltip=f"Click to download {filename}",
        layout=widgets.Layout(width='auto', margin='10px 0px')
    )

    def on_download_clicked(b):
        files.download(filename)

    download_button.on_click(on_download_clicked)
    display(download_button)

# 5. The Execution Loop
def run_translator_agent():
    queue_file = "selected_for_translation.csv"
    if not os.path.exists(queue_file):
        print("No approved works found in the queue. Have you run the Intercept cell?")
        return

    df = pd.read_csv(queue_file)

    # --- Enforce the 90% threshold ---
    df['Score'] = pd.to_numeric(df['Score'], errors='coerce')
    df = df[df['Score'] >= 90]

    if df.empty:
        print("No works in the queue currently meet the strict 90% threshold.")
        return
    # ---------------------------------------------------

    for index, row in df.iterrows():
        title = row['Title']
        score = row['Score']

        # Read ONLY the exact extract saved by the Reviewer
        target_text = str(row.get('Extracted_Text', ''))

        print(f"\n--- Initiating Agent 2 for: '{title}' (Score: {score}%) ---")

        # Catch empty extracts or NaN values from Pandas
        if len(target_text) < 100 or target_text.lower() == 'nan':
            print("No valid extracted ghost story found for translation. Skipping.")
            continue

        # Chunk ONLY the extracted article, not the whole magazine
        chunks = chunk_text(target_text)
        print(f"Extracted story broken into {len(chunks)} manageable chunks.")

        # Translate sequentially
        translated_chunks = []
        for i, chunk in enumerate(chunks):
            print(f"Processing chunk {i+1}/{len(chunks)}")
            translated_text = translate_text(chunk)
            translated_chunks.append(translated_text)
            time.sleep(3) # Polite API pause

        # Package it
        publish_to_epub(title, translated_chunks)

# --- EXECUTION ---
run_translator_agent()


--- Initiating Agent 2 for: 'The Strand Magazine: Vol. 07, Issue 37, January, 1894. by Various' (Score: 95%) ---
Extracted story broken into 13 manageable chunks.
Processing chunk 1/13
Translating chunk...
Processing chunk 2/13
Translating chunk...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 57.946485396s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-fla